# Data Preprocessing Pipeline
**ContextFlow AI** - Enterprise Knowledge Assistant

Notebook ini melakukan:
1. Load dataset dari HuggingFace
2. Text cleaning & normalization
3. Format ke instruction style
4. Deduplicate & filter
5. Train/Val split
6. Simpan ke disk

In [1]:
import sys
import os
sys.path.append('..')

import pandas as pd
import numpy as np
from datasets import load_dataset, concatenate_datasets, Dataset
from app.utils.logger import logger
from app.utils.config import config
from app.preprocessing.cleaner import clean_text, remove_duplicates, remove_missing, filter_by_length, filter_by_length_per_source
from app.preprocessing.formatter import apply_formatter, INSTRUCTION_TEMPLATE

print('All imports OK')

d:\bootcamp\final_project_fine_tuning\contextflow_ai_fine_tuning\venv311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


All imports OK


In [2]:
import shutil, os

cache_path = os.path.expanduser("~/.cache/huggingface/datasets")
for folder in os.listdir(cache_path):
    if "open_assistant" in folder.lower() or "oasst" in folder.lower():
        shutil.rmtree(os.path.join(cache_path, folder))
        print(f"Deleted: {folder}")

Deleted: OpenAssistant___oasst1


## 1. Load Datasets dari HuggingFace

In [3]:
# Import semua loader dari module yang sudah ada
from app.preprocessing.data_loader import (
    load_dolly_15k,      # loader Dolly 15K
    load_alpaca,         # loader Alpaca
    load_openassistant,  # loader OpenAssistant
    load_coqa,           # loader CoQA — sudah include flatten per Q&A pair
)

# Load semua dataset via fungsi loader masing-masing
# Jangan pakai load_dataset() langsung karena CoQA butuh flatten dulu
dolly  = load_dolly_15k()     # kolom: instruction, context, response
alpaca = load_alpaca()         # kolom: instruction, input, output
oasst  = load_openassistant()  # kolom: message_id, text, role, dll
coqa   = load_coqa()           # kolom: question, context, answer (sudah di-flatten)

# Batasi jumlah sample per dataset
SAMPLE_SIZE = 10000

raw_datasets = {
    "dolly":         dolly.select(range(min(SAMPLE_SIZE, len(dolly)))),
    "alpaca":        alpaca.select(range(min(SAMPLE_SIZE, len(alpaca)))),
    "openassistant": oasst.select(range(min(SAMPLE_SIZE, len(oasst)))),
    "coqa":          coqa.select(range(min(SAMPLE_SIZE, len(coqa)))),  # pakai coqa, bukan coqa_ds
}

# Verifikasi kolom tiap dataset sudah benar sebelum lanjut ke formatter
for name, ds in raw_datasets.items():
    print(f"{name}: {len(ds)} samples | columns: {ds.column_names}")

2026-05-17 21:58:46 | INFO | app.preprocessing.data_loader:8 - Loading Dolly 15K dataset...
2026-05-17 21:58:51 | INFO | app.preprocessing.data_loader:10 - Dolly 15K loaded: 15011 samples
2026-05-17 21:58:51 | INFO | app.preprocessing.data_loader:15 - Loading Alpaca dataset...
2026-05-17 21:58:56 | INFO | app.preprocessing.data_loader:17 - Alpaca loaded: 52002 samples
2026-05-17 21:58:56 | INFO | app.preprocessing.data_loader:35 - Loading OpenAssistant dataset...


KeyboardInterrupt: 

## 2. Sampling & Formatting

In [ ]:
SAMPLE_SIZE = 10000

raw_datasets = {
    'dolly': dolly.select(range(min(SAMPLE_SIZE, len(dolly)))),
    'alpaca': alpaca.select(range(min(SAMPLE_SIZE, len(alpaca)))),
    'openassistant': oasst.select(range(min(SAMPLE_SIZE, len(oasst)))),
    'coqa': coqa.select(range(min(SAMPLE_SIZE, len(coqa)))),
}

for name, ds in raw_datasets.items():
    print(f'{name}: {len(ds)} samples')

dolly: 10000 samples
alpaca: 10000 samples
openassistant: 10000 samples
coqa: 10000 samples


In [ ]:
print(raw_datasets["coqa"][0])
print(raw_datasets["coqa"].column_names)

{'question': 'When was the Vat formally opened?', 'context': 'The Vatican Apostolic Library (), more commonly called the Vatican Library or simply the Vat, is the library of the Holy See, located in Vatican City. Formally established in 1475, although it is much older, it is one of the oldest libraries in the world and contains one of the most significant collections of historical texts. It has 75,000 codices from throughout history, as well as 1.1 million printed books, which include some 8,500 incunabula. \n\nThe Vatican Library is a research library for history, law, philosophy, science and theology. The Vatican Library is open to anyone who can document their qualifications and research needs. Photocopies for private study of pages from books published between 1801 and 1990 can be requested in person or by mail. \n\nIn March 2014, the Vatican Library began an initial four-year project of digitising its collection of manuscripts, to be made available online. \n\nThe Vatican Secret A

In [ ]:
# Format semua dataset ke instruction style
formatted = []
for source, ds in raw_datasets.items():
    try:
        fmt = apply_formatter(ds, source)
        formatted.append(fmt)
        print(f'{source}: {len(fmt)} formatted')
    except Exception as e:
        print(f'Error formatting {source}: {e}')

combined = concatenate_datasets(formatted)
print(f'\nTotal combined: {len(combined)} samples')

2026-05-15 19:24:41 | INFO | app.preprocessing.formatter:157 - Formatted 10000 samples from dolly
dolly: 10000 formatted
2026-05-15 19:24:41 | INFO | app.preprocessing.formatter:157 - Formatted 10000 samples from alpaca
alpaca: 10000 formatted


Map: 100%|██████████| 10000/10000 [00:00<00:00, 13366.66 examples/s]

2026-05-15 19:24:42 | INFO | app.preprocessing.formatter:157 - Formatted 10000 samples from openassistant


openassistant: 10000 formatted


Map: 100%|██████████| 10000/10000 [00:00<00:00, 14267.16 examples/s]

2026-05-15 19:24:43 | INFO | app.preprocessing.formatter:157 - Formatted 10000 samples from coqa
coqa: 10000 formatted

Total combined: 40000 samples


In [ ]:
# Lihat contoh hasil format
print(combined[0]['text'])

### Instruction:
When did Virgin Australia start operating?

### Input:
Virgin Australia, the trading name of Virgin Australia Airlines Pty Ltd, is an Australian-based airline. It is the largest airline by fleet size to use the Virgin brand. It commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route. It suddenly found itself as a major airline in Australia's domestic market after the collapse of Ansett Australia in September 2001. The airline has since grown to directly serve 32 cities in Australia, from hubs in Brisbane, Melbourne and Sydney.

### Response:
Virgin Australia commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route.


In [ ]:
# Cek distribusi output kosong per source
df = combined.to_pandas()
print(df.groupby('source')['output'].apply(lambda x: (x.str.strip() == "").sum()))

source
alpaca           7
coqa             0
dolly            0
openassistant    0
Name: output, dtype: int64


In [ ]:
# Sebelum combined.to_pandas(), cek hasil formatter CoQA
coqa_formatted = raw_datasets["coqa"]
fmt = apply_formatter(coqa_formatted, "coqa")
print(fmt[0])                        # lihat semua kolom hasil formatter
print(repr(fmt[0].get("output")))    # lihat exact value output
print(fmt.column_names)              # pastiin kolom 'output' ada

Map: 100%|██████████| 10000/10000 [00:00<00:00, 14743.33 examples/s]

2026-05-15 19:24:44 | INFO | app.preprocessing.formatter:157 - Formatted 10000 samples from coqa
{'instruction': 'When was the Vat formally opened?', 'input': 'The Vatican Apostolic Library (), more commonly called the Vatican Library or simply the Vat, is the library of the Holy See, located in Vatican City. Formally established in 1475, although it is much older, it is one of the oldest libraries in the world and contains one of the most significant collections of historical texts. It has 75,000 codices from throughout history, as well as 1.1 million printed books, which include some 8,500 incunabula. The Vatican Library is a research library for h', 'output': 'It was formally established in 1475', 'text': '### Instruction:\nWhen was the Vat formally opened?\n\n### Input:\nThe Vatican Apostolic Library (), more commonly called the Vatican Library or simply the Vat, is the library of the Holy See, located in Vatican City. Formally established in 1475, although it is much older, it is 

## 3. Text Cleaning

In [ ]:
from app.preprocessing.cleaner import (
    remove_missing,           # hapus baris dengan nilai kosong
    remove_duplicates,        # hapus duplikat
    filter_by_length_per_source,  # filter panjang secara source-aware
)

# Konversi combined HuggingFace Dataset → pandas DataFrame untuk cleaning
df = combined.to_pandas()
print(f"Before cleaning: {len(df)} rows")
print(df.groupby("source").size())  # lihat distribusi awal per source

# === STEP 1: Hapus missing values ===
# Kolom 'output' untuk openassistant sengaja kosong (by design), tidak ikut difilter
# Kolom 'instruction' wajib ada untuk semua source
df = remove_missing(df, required_cols=["instruction", "output"])

# === STEP 2: Hapus duplikat ===
# Kombinasi instruction + output yang identik dianggap duplikat
df = remove_duplicates(df, subset=["instruction", "output"])

# === STEP 3: Filter panjang secara source-aware ===
# Setiap source punya threshold berbeda (lihat LENGTH_CONFIG di cleaner.py):
# - CoQA    : min=1   karena jawaban memang 1-5 kata ("research", "1475", "five")
# - Dolly   : min=10  karena response deskriptif panjang
# - Alpaca  : min=10  sama seperti dolly
# - OA      : min=0   output kosong by design, jangan difilter
# - Company : min=5   fleksibel
df = filter_by_length_per_source(df, col="output")

# === STEP 4: Filter panjang instruction (berlaku semua source) ===
# Instruction minimal 5 karakter — terlalu pendek tidak bermakna
df = df[df["instruction"].str.len() >= 5]

# Reset index setelah semua filter
df = df.reset_index(drop=True)

print(f"\nAfter cleaning: {len(df)} rows")
print(df.groupby("source").size())  # verifikasi semua source masih ada

Before cleaning: 40000 rows
source
alpaca           10000
coqa             10000
dolly            10000
openassistant    10000
dtype: int64
2026-05-15 19:24:44 | INFO | app.preprocessing.cleaner:60 - Removed 10 missing/empty rows
2026-05-15 19:24:44 | INFO | app.preprocessing.cleaner:28 - Removed 39 duplicates
2026-05-15 19:24:44 | INFO | app.preprocessing.cleaner:116 - [dolly] filter 'output': min=10, max=1500 → 9399 rows
2026-05-15 19:24:44 | INFO | app.preprocessing.cleaner:116 - [alpaca] filter 'output': min=10, max=1500 → 9423 rows
2026-05-15 19:24:44 | INFO | app.preprocessing.cleaner:116 - [openassistant] filter 'output': min=0, max=9999 → 9992 rows
2026-05-15 19:24:44 | INFO | app.preprocessing.cleaner:116 - [coqa] filter 'output': min=1, max=500 → 9978 rows
2026-05-15 19:24:44 | INFO | app.preprocessing.cleaner:139 - Total after filter_by_length_per_source: 38792 rows

After cleaning: 38527 rows
source
alpaca           9423
coqa             9758
dolly            9399
openassis

In [ ]:
# Contoh data bersih
df[['instruction', 'output', 'source']].head(10)

,instruction,output,source
0,When did Virgin Australia start operating?,Virgin Australia commenced services on 31 Augu...,dolly
1,Why can camels survive for long without water?,Camels use the fat in their humps to keep them...,dolly
2,"Alice's parents have three daughters: Amy, Jes...",The name of the third daughter is Alice,dolly
3,When was Tomoaki Komorida born?,"Tomoaki Komorida was born on July 10,1981.",dolly
4,If I have more pieces at the time of stalemate...,No. Stalemate is a drawn position. It doesn't ...,dolly
5,"Given a reference text about Lollapalooza, whe...",Lollapalooze is an annual musical festival hel...,dolly
6,Who gave the UN the land in NY to build their HQ,John D Rockerfeller,dolly
7,Why mobile is bad for human,We are always engaged one phone which is not g...,dolly
8,Who was John Moses Browning?,John Moses Browning is one of the most well-kn...,dolly
9,Who was Kyle Van Zyl playing against when he s...,Kyle Van Zyl was playing against Boland U21 wh...,dolly


In [ ]:
df['source'].unique()

array(['dolly', 'alpaca', 'openassistant', 'coqa'], dtype=object)

## 4. Train/Val Split

In [ ]:
split_idx = int(len(df) * 0.9)
train_df = df.iloc[:split_idx]
val_df = df.iloc[split_idx:]

print(f'Train: {len(train_df)} samples')
print(f'Val: {len(val_df)} samples')

train_ds = Dataset.from_pandas(train_df, preserve_index=False)
val_ds = Dataset.from_pandas(val_df, preserve_index=False)

Train: 34674 samples
Val: 3853 samples


## 5. Simpan ke Disk

In [ ]:
os.makedirs(config.FORMATTED_DATA_DIR, exist_ok=True)

train_ds.save_to_disk(os.path.join(config.FORMATTED_DATA_DIR, 'train'))
val_ds.save_to_disk(os.path.join(config.FORMATTED_DATA_DIR, 'val'))

print(f'Saved train ({len(train_ds)}) and val ({len(val_ds)}) to {config.FORMATTED_DATA_DIR}')

Saving the dataset (1/1 shards): 100%|██████████| 3853/3853 [00:00<00:00, 394489.41 examples/s]

Saved train (34674) and val (3853) to D:\bootcamp\final_project_fine_tuning\contextflow_ai_fine_tuning\data\formatted


In [ ]:
# Verifikasi
from datasets import load_from_disk
train_check = load_from_disk(os.path.join(config.FORMATTED_DATA_DIR, 'train'))
print(f'Loaded train: {len(train_check)} samples')
print(train_check[0])

Loaded train: 34674 samples
{'instruction': 'When did Virgin Australia start operating?', 'input': "Virgin Australia, the trading name of Virgin Australia Airlines Pty Ltd, is an Australian-based airline. It is the largest airline by fleet size to use the Virgin brand. It commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route. It suddenly found itself as a major airline in Australia's domestic market after the collapse of Ansett Australia in September 2001. The airline has since grown to directly serve 32 cities in Australia, from hubs in Brisbane, Melbourne and Sydney.", 'output': 'Virgin Australia commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route.', 'text': "### Instruction:\nWhen did Virgin Australia start operating?\n\n### Input:\nVirgin Australia, the trading name of Virgin Australia Airlines Pty Ltd, is an Australian-based airline. It is the largest airline by fleet size to use the Virgin brand. It commenced